In [2]:
# import required Libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

print("TF:", tf.__version__)
print("PD:", pd.__version__)

TF: 2.19.0
PD: 2.2.2


In [3]:
# Step 1: Load your actual telco.csv (attached dataset)

csv_path = "telco.csv"   # make sure this matches the uploaded filename

df = pd.read_csv(csv_path)

print("Shape:", df.shape)         # (rows, columns)
print("\nFirst 5 rows:")
print(df.head())

print("\nInfo:")
print(df.info())

print("\nChurn Label value counts:")
print(df["Churn Label"].value_counts())

Shape: (7043, 50)

First 5 rows:
  Customer ID  Gender  Age Under 30 Senior Citizen Married Dependents  \
0  8779-QRDMV    Male   78       No            Yes      No         No   
1  7495-OOKFY  Female   74       No            Yes     Yes        Yes   
2  1658-BYGOY    Male   71       No            Yes      No        Yes   
3  4598-XLKNJ  Female   78       No            Yes     Yes        Yes   
4  4846-WHAFZ  Female   80       No            Yes     Yes        Yes   

   Number of Dependents        Country       State  ...  \
0                     0  United States  California  ...   
1                     1  United States  California  ...   
2                     3  United States  California  ...   
3                     1  United States  California  ...   
4                     1  United States  California  ...   

  Total Extra Data Charges  Total Long Distance Charges  Total Revenue  \
0                       20                         0.00          59.65   
1                        

In [4]:
# Step 2: Create numeric target and drop ID + text label

target_col = "Churn"  # name for our numeric target column

# 1) Create numeric 'Churn' from text 'Churn Label' ("Yes"/"No" -> 1/0)
# we need to convert strings to integers for the model.
df[target_col] = df["Churn Label"].map({"Yes": 1, "No": 0}).astype("int32")

# 2) Drop 'Customer ID' (identifier, not a useful feature) AND 'Churn Label' (text version of target)
# we now explicitly drop 'Churn Label' so it is NOT used as a feature.
df = df.drop(columns=["Customer ID", "Churn Label"])

In [5]:
df.head()

,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,City,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Score,CLTV,Churn Category,Churn Reason,Churn
0,Male,78,No,Yes,No,No,0,United States,California,Los Angeles,...,20,0.00,59.65,3,Churned,91,5433,Competitor,Competitor offered more data,1
1,Female,74,No,Yes,Yes,Yes,1,United States,California,Los Angeles,...,0,390.80,1024.10,3,Churned,69,5302,Competitor,Competitor made better offer,1
2,Male,71,No,Yes,No,Yes,3,United States,California,Los Angeles,...,0,203.94,1910.88,2,Churned,81,3179,Competitor,Competitor made better offer,1
3,Female,78,No,Yes,Yes,Yes,1,United States,California,Inglewood,...,0,494.00,2995.07,2,Churned,88,5337,Dissatisfaction,Limited range of services,1
4,Female,80,No,Yes,Yes,Yes,1,United States,California,Whittier,...,0,234.21,3102.36,2,Churned,67,2793,Price,Extra data charges,1


In [6]:
df[target_col].value_counts()

,count
Churn,
0,5174
1,1869


In [7]:
# Step 3: Identify numeric and categorical columns

# Numeric columns: int64/float64, EXCLUDING the target
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target_col]

# Categorical columns: object (string) columns
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("\nCategorical columns:", categorical_cols)

Numeric columns: ['Age', 'Number of Dependents', 'Zip Code', 'Latitude', 'Longitude', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue', 'Satisfaction Score', 'Churn Score', 'CLTV']

Categorical columns: ['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Country', 'State', 'City', 'Quarter', 'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Customer Status', 'Churn Category', 'Churn Reason']


In [8]:
# Step 4: Handle missing values BEFORE encoding
# ----------------------------------------------------------------
# Numeric columns:
#   - fill NaN with the column median (simple, robust choice).
# Categorical columns:
#   - fill NaN with a special category "Unknown" so get_dummies can handle it.

for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

In [9]:
# Step 5: One-hot encode categorical features
#   - get_dummies turns each category into separate 0/1 columns.

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Encoded shape:", df_encoded.shape)

Encoded shape: (7043, 1182)


In [18]:
df_encoded.head()

,Age,Number of Dependents,Zip Code,Latitude,Longitude,Population,Number of Referrals,Tenure in Months,Avg Monthly Long Distance Charges,Avg Monthly GB Download,...,Churn Reason_Limited range of services,Churn Reason_Long distance charges,Churn Reason_Moved,Churn Reason_Network reliability,Churn Reason_Poor expertise of online support,Churn Reason_Poor expertise of phone support,Churn Reason_Price too high,Churn Reason_Product dissatisfaction,Churn Reason_Service dissatisfaction,Churn Reason_Unknown
0,78,0,90022,34.023810,-118.156582,68701,0,1,0.00,8,...,False,False,False,False,False,False,False,False,False,False
1,74,1,90063,34.044271,-118.185237,55668,1,8,48.85,17,...,False,False,False,False,False,False,False,False,False,False
2,71,3,90065,34.108833,-118.229715,47534,0,18,11.33,52,...,False,False,False,False,False,False,False,False,False,False
3,78,1,90303,33.936291,-118.332639,27778,1,25,19.76,12,...,True,False,False,False,False,False,False,False,False,False
4,80,1,90602,33.972119,-118.020188,26265,1,37,6.33,14,...,False,False,False,False,False,False,False,False,False,False


In [10]:
# Step 6: Split into features X and target y
X = df_encoded.drop(columns=[target_col]).values.astype("float32")
y = df_encoded[target_col].values.astype("int32")

print("X shape:", X.shape)  # (num_samples, num_features_after_encoding)
print("y shape:", y.shape)  # (num_samples,)

X shape: (7043, 1181)
y shape: (7043,)


In [11]:
from sklearn.model_selection import train_test_split

# Step 7: Train/test split (same concept as Day 3)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # keep churn proportion similar in train/test sets
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (5634, 1181) (5634,)
Test shape: (1409, 1181) (1409,)


In [12]:
from sklearn.preprocessing import StandardScaler

# Step 8: Scale features with StandardScaler (fit on train, apply to train + test)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled train mean (first 5 features):", X_train_scaled.mean(axis=0)[:5])
print("Scaled train std  (first 5 features):", X_train_scaled.std(axis=0)[:5])

Scaled train mean (first 5 features): [-2.7347889e-08  2.9969479e-07 -3.4383225e-09  1.5255573e-08
  6.3899903e-09]
Scaled train std  (first 5 features): [0.999998  0.9999897 1.        0.9999992 1.0000002]


In [22]:
# Step 9: Build the Keras model for churn prediction

input_dim = X_train_scaled.shape[1]  # number of input features after encoding

inputs = tf.keras.Input(shape=(input_dim,))  # input is a vector of length input_dim

# Hidden layer 1: more neurons to handle the larger feature space
x = layers.Dense(64, activation="relu")(inputs)

# Hidden layer 2: another Dense layer
x = layers.Dense(32, activation="relu")(x)

# Output layer: 1 neuron with sigmoid activation
#  - outputs a value in [0, 1] interpreted as probability that Churn = 1.
outputs = layers.Dense(1, activation="sigmoid")(x)

churn_model = tf.keras.Model(inputs=inputs, outputs=outputs)

churn_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1181)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        75,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,761 (303.75 KB)

 Trainable params: 77,761 (303.75 KB)

 Non-trainable params: 0 (0.00 B)

**We can also use the Sequential Model(Container) intead of the Functional API**

In [13]:
input_dim = X_train_scaled.shape[1]

churn_model = tf.keras.Sequential([
    layers.Input(shape=(input_dim, )),

    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
churn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │        75,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,761 (303.75 KB)

 Trainable params: 77,761 (303.75 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# Step 10: Compile the model
# Same as Day 3 conceptually: binary cross-entropy + accuracy.

churn_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"]
)

# callbacks: EarlyStopping and ModelCheckpoint

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# EarlyStopping:
#  - monitor='val_loss' -> watch validation loss each epoch
#  - patience=3         -> if val_loss does NOT improve for 3 epochs in a row, stop training early
#  - restore_best_weights=True -> after stopping, revert to the epoch with the lowest val_loss.
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# ModelCheckpoint:
#  - saves the model to a file whenever val_loss improves
#  - save_best_only=True -> only keep the best model (lowest val_loss) on disk.
checkpoint = ModelCheckpoint(
    "best_churn_model.keras",
    monitor="val_loss",
    save_best_only=True
)

# Train the model
history_churn = churn_model.fit(
    X_train_scaled,
    y_train,
    epochs=30,          # maximum number of epochs; EarlyStopping may stop earlier
    batch_size=64,      # # model will look at 64 training examples at a time before doing one weight update.
    validation_split=0.2,
    callbacks=[early_stop, checkpoint],  # pass callbacks here
    verbose=2
)

Epoch 1/30
71/71 - 3s - 38ms/step - accuracy: 0.8926 - loss: 0.2895 - val_accuracy: 0.9787 - val_loss: 0.0844
Epoch 2/30
71/71 - 1s - 9ms/step - accuracy: 0.9962 - loss: 0.0224 - val_accuracy: 0.9876 - val_loss: 0.0398
Epoch 3/30
71/71 - 0s - 7ms/step - accuracy: 0.9998 - loss: 0.0042 - val_accuracy: 0.9885 - val_loss: 0.0330
Epoch 4/30
71/71 - 0s - 6ms/step - accuracy: 1.0000 - loss: 0.0019 - val_accuracy: 0.9885 - val_loss: 0.0303
Epoch 5/30
71/71 - 0s - 6ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.9885 - val_loss: 0.0287
Epoch 6/30
71/71 - 0s - 6ms/step - accuracy: 1.0000 - loss: 7.0502e-04 - val_accuracy: 0.9894 - val_loss: 0.0278
Epoch 7/30
71/71 - 0s - 6ms/step - accuracy: 1.0000 - loss: 4.9622e-04 - val_accuracy: 0.9902 - val_loss: 0.0271
Epoch 8/30
71/71 - 0s - 6ms/step - accuracy: 1.0000 - loss: 3.6967e-04 - val_accuracy: 0.9911 - val_loss: 0.0267
Epoch 9/30
71/71 - 0s - 6ms/step - accuracy: 1.0000 - loss: 2.8010e-04 - val_accuracy: 0.9911 - val_loss: 0.0262
Ep

In [15]:
# Step 11: Evaluate on the test set

test_loss, test_acc = churn_model.evaluate(X_test_scaled, y_test, verbose=2)
print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

45/45 - 0s - 3ms/step - accuracy: 0.9872 - loss: 0.0370
Test loss: 0.037005070596933365
Test accuracy: 0.9872249960899353


In [16]:
# Step 12: Predict churn for a manually specified customer
# ------------------------------------------------------

#  - The model was trained on df_encoded (after filling NaNs, one-hot, scaling, etc.).
#  - For a new customer, we must apply the SAME preprocessing steps and column order.
#    Means, pass it through the same pipeline.

# 12.1: Create a single-row DataFrame starting from an existing row
#     (this guarantees all columns exist, with correct names and types).
sample_original = df.iloc[[0]].copy()  # double brackets -> keep it as DataFrame (1 row)

# Now modify some values to simulate a new customer.
sample_original["Age"] = 30
sample_original["Tenure in Months"] = 5
sample_original["Monthly Charge"] = 80.0
sample_original["Contract"] = "Month-to-Month"
sample_original["Internet Type"] = "Fiber Optic"
sample_original["Offer"] = "Offer E"

print("Sample original row (before encoding):")
print(sample_original)

# 12.2: Apply the SAME preprocessing as training:
#     - fill numeric NaNs with median
#     - fill categorical NaNs with "Unknown"
#     - one-hot encode with SAME columns as df_encoded
#     - scale with the SAME scaler (already fitted)

# a) Handle missing values for this single row
sample_numeric = sample_original[numeric_cols].copy()
sample_categorical = sample_original[categorical_cols].copy()

for col in numeric_cols:
    median_val = df[col].median()  # use TRAINING df median
    sample_numeric[col] = sample_numeric[col].fillna(median_val)

for col in categorical_cols:
    sample_categorical[col] = sample_categorical[col].fillna("Unknown")

# Recombine numeric + categorical for this row
sample_clean = pd.concat([sample_numeric, sample_categorical, sample_original[[target_col]]], axis=1)

# b) One-hot encode categoricals for this row
sample_encoded = pd.get_dummies(sample_clean, columns=categorical_cols, drop_first=True)

# c) Align columns with training df_encoded
#    - We reindex columns to match df_encoded's columns.
#    - Any missing column is filled with 0 (meaning that category is not present for this row).
#    - We then drop the target column to get features X.

sample_encoded_aligned = sample_encoded.reindex(columns=df_encoded.columns, fill_value=0)
sample_X = sample_encoded_aligned.drop(columns=[target_col]).values.astype("float32")

# d) Scale using the SAME scaler as training
sample_X_scaled = scaler.transform(sample_X)

# 8.3: Predict with the trained model

sample_prob = churn_model.predict(sample_X_scaled)[0, 0]  # single scalar probability
sample_pred_label = int(sample_prob >= 0.5)  # 0 or 1

print("\nPredicted churn probability:", float(sample_prob))
print("Predicted churn label (0 = No, 1 = Yes):", sample_pred_label)

if sample_pred_label == 1:
    print("Model prediction: This customer is LIKELY to churn (Yes).")
else:
    print("Model prediction: This customer is NOT likely to churn (No).")

Sample original row (before encoding):
  Gender  Age Under 30 Senior Citizen Married Dependents  \
0   Male   30       No            Yes      No         No   

   Number of Dependents        Country       State         City  ...  \
0                     0  United States  California  Los Angeles  ...   

   Total Extra Data Charges  Total Long Distance Charges  Total Revenue  \
0                        20                          0.0          59.65   

   Satisfaction Score Customer Status Churn Score  CLTV  Churn Category  \
0                   3         Churned          91  5433      Competitor   

                   Churn Reason Churn  
0  Competitor offered more data     1  

[1 rows x 49 columns]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step

Predicted churn probability: 0.9996805787086487
Predicted churn label (0 = No, 1 = Yes): 1
Model prediction: This customer is LIKELY to churn (Yes).
